In [23]:
import json
import tensorflow as tf
from tensorflow.keras.models import load_model

print("Targeting GPU for direct model loading...")

# Force TensorFlow to process and hold the direct model compilation inside your GPU
with tf.device('/GPU:0'):
    
    # 1. Directly load the entire model file (Structure + Optimizer state + Weights)
    model_v3 = load_model("daily_dialog_lstm_v2.keras")
    print("✔ Entire .keras model loaded directly into GPU memory!")

# 2. Load your vocabulary tracking setup back into system memory
with open("vocab_config.json", "r") as f:
    saved_vocab = json.load(f)

# 3. Synchronize your vectorization layer
vectorize_layer = tf.keras.layers.TextVectorization(vocabulary=saved_vocab)
print("✔ Vocabulary and text vectorizer synchronized perfectly.")

Targeting GPU for direct model loading...
✔ Entire .keras model loaded directly into GPU memory!
✔ Vocabulary and text vectorizer synchronized perfectly.


In [24]:
model_v3.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 8, 256)            3840000   
                                                                 
 lstm_2 (LSTM)               (None, 8, 512)            1574912   
                                                                 
 dropout_2 (Dropout)         (None, 8, 512)            0         
                                                                 
 lstm_3 (LSTM)               (None, 512)               2099200   
                                                                 
 dropout_3 (Dropout)         (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 15000)             7695000   
                                                                 
Total params: 15,209,112
Trainable params: 15,209,112


In [3]:
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from datasets import load_dataset





C:\Users\VICTUS\anaconda3\envs\gpu_room\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_dataset = load_dataset("parquet", data_files={
    "train": "hf://datasets/daily_dialog@refs/convert/parquet/default/train/0000.parquet",
    "validation": "hf://datasets/daily_dialog@refs/convert/parquet/default/validation/0000.parquet",
    "test": "hf://datasets/daily_dialog@refs/convert/parquet/default/test/0000.parquet"
})

In [27]:
import numpy as np

def generate_text_stream(seed_phrase, words_to_predict=12, temperature=0.3):
    """
    Takes a starter seed phrase, predicts the next word, appends it, 
    and loops to generate a full conversational line of text.
    """
    generated_text = seed_phrase
    print(f"Seed Prompt: '{seed_phrase}'")
    print(f"Generated Chat: {seed_phrase}", end=" ")

    for _ in range(words_to_predict):
        # 1. Map raw input text directly to integer tokens via your adapted layer
        tokens = vectorize_layer([generated_text]).numpy()[0]
        
        # Remove trailing/leading padding from the vectorization matrix
        clean_tokens = tokens[tokens != 0]
        
        # 2. Enforce structural sequence constraint of exactly 8 elements
        if len(clean_tokens) < 8:
            # Pre-pad with zeros if the sentence is shorter than context window
            padded_input = np.pad(clean_tokens, (8 - len(clean_tokens), 0), 'constant')
        else:
            # Truncate to take only the last 8 words if context length is exceeded
            padded_input = clean_tokens[-8:]
            
        # Reshape to fit model batch dimensions: (1, 8)
        input_tensor = np.array([padded_input])
        
        # 3. Generate predictions
        predictions = model_v3.predict(input_tensor, verbose=0)[0]
        
        # 4. Temperature Sampling (Prevents the model from getting stuck repeating words)
        # 0.6 is a good sweet spot for clean grammar with a bit of variety
        predictions = np.log(predictions + 1e-7) / temperature
        exp_preds = np.exp(predictions)
        predicted_probs = exp_preds / np.sum(exp_preds)
        
        # Sample the next token index based on the calculated probability distribution
        predicted_idx = np.random.choice(len(predicted_probs), p=predicted_probs)
        
        # 5. Reverse map the index back to a string word
        vocabulary = vectorize_layer.get_vocabulary()
        predicted_word = vocabulary[predicted_idx]
        
        # Break execution if blanks or structural flags are generated
        if predicted_word in ["", "[UNK]"]:
            continue
            
        # Append word to history for the next iteration loop pass
        generated_text += " " + predicted_word
        print(predicted_word, end=" ", flush=True)
    print("\n" + "="*50)

In [31]:
generate_text_stream("I would like to change rooms if possible")
generate_text_stream("Yes , on the shippers side , when the cargos arrives , all relevant documents will be ")
generate_text_stream(" you take your whipped up eggs , or whisked up eggs")

Seed Prompt: 'I would like to change rooms if possible'
Generated Chat: I would like to change rooms if possible i can do it in the morning or a little bit of 
Seed Prompt: 'Yes , on the shippers side , when the cargos arrives , all relevant documents will be '
Generated Chat: Yes , on the shippers side , when the cargos arrives , all relevant documents will be  forwarded to the shower you should be 
Seed Prompt: ' you take your whipped up eggs , or whisked up eggs'
Generated Chat:  you take your whipped up eggs , or whisked up eggs and pour them into the pan and it ’ s a very 
